# 04 · Which garment reference works best? (Colab T4, ~1 h)
Runs **stock CatVTON** on the same 24 worn pairs (test + val splits) once per garment version and scores each result against the **real photo** of the model wearing that outfit:

| variant | garment image given to CatVTON |
|---|---|
| `full` | the other model photo, exactly as on the shop page |
| `faceless` | same photo, head greyed out |
| `cutout` | garment only, cut out with SCHP |
| `flat` | brand flat lay (only ETHNC SKUs that have one) |

Scores are measured **inside the try-on mask only**: SSIM ↑, LPIPS ↓, and `hem_gap_px` ↓ (how far the generated kameez hem is from the real hem).
Run **after notebook 03** (it needs the agnostic masks and `cloth_cutout/`). The winner goes into `docs/decisions.md` and becomes `--worn-cloth` for the final export.

## 0 · Setup
Mount Drive and get the code. Put **either** your GitHub URL in `REPO_URL`, **or** leave it empty to use `desi-fit-ai.zip` from the dataset folder on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
ROOT   = '/content/drive/MyDrive/AainaClosetDataset'      # scraped images live here
BUILD  = f'{ROOT}/_build'                                  # all intermediate CSVs + review pages
DATASET= '/content/drive/MyDrive/DesiFitAI/processed/DesiFit-VTON-v1'   # final exported dataset
REPO_URL = ''   # e.g. 'https://github.com/<you>/desi-fit-ai.git'  (private repo: use a token URL)
os.makedirs(BUILD, exist_ok=True)

if REPO_URL:
    !rm -rf /content/desi-fit-ai && git clone -q $REPO_URL /content/desi-fit-ai
else:
    !rm -rf /content/desi-fit-ai && unzip -q "$ROOT/desi-fit-ai.zip" -d /content/
%cd /content/desi-fit-ai
!git log -1 --oneline 2>/dev/null || ls

### CatVTON + fixes (same as notebook 03). Restart the session after this cell if `huggingface_hub` was re-installed.

In [ ]:
!git clone -q https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON || true
!grep -v -E "^(torch|torchvision|xformers)" /content/CatVTON/requirements.txt > /content/req_colab.txt
!pip -q install -r /content/req_colab.txt
!pip -q install huggingface_hub==0.25.2 lpips scikit-image opencv-python-headless pandas

### Run (resumable: finished images are re-used)
384×512 keeps it to ~10 s per image on a T4. Increase `n` once you trust it.

In [ ]:
import sys; sys.path.insert(0, '/content/desi-fit-ai')
from data_tools.compare_modes import run, summary, sheet
OUT = '/content/drive/MyDrive/DesiFitAI/results/compare_modes'
res = run(DATASET, '/content/CatVTON', OUT, n=24, width=384, height=512)
per_variant, fair = summary(res)
print('All pairs:'); display(per_variant)
print('Only pairs where every variant exists (use this one):'); display(fair)

### Look at the outputs — the numbers are not enough on their own

In [ ]:
from IPython.display import Image as IImage
IImage(sheet(res, DATASET, OUT, '/content/compare_sheet.png', n=8))

### Decide
- Pick the variant with the best LPIPS **and** hem gap, and confirm by eye on the sheet.
- If `faceless` ≈ `full`, prefer `faceless` (no identity leakage).
- Write the result into `docs/decisions.md`, then re-run the export cell of notebook 02 with `--worn-cloth <winner> --overwrite`.
- This table, plus `flat` vs. the worn variants, is a results-chapter finding: *does a shop's model photo work as well as a flat lay as the garment input?*